# Meta-RAG: Using Retrieval to Evaluate Retrieval

Standard RAG evaluation asks: does the answer match the gold label?
This requires a human-written gold answer for every question.

This project asks a different question:
**Can a second retrieval layer evaluate whether the first layer's answer
is specifically supported by the retrieved context, without needing gold labels?**

This is called meta-evaluation: using the same kind of system
to judge the outputs of itself.

The key idea: if an answer is genuinely grounded in the retrieved passage,
it should overlap with that passage MORE than it overlaps with any other
independently retrieved passage. We call this the specificity ratio.

Dataset: SQuAD 2.0 dev set. 200 answerable questions evaluated.
No gold labels used in the meta-evaluation layer.
Gold labels used only at the end to validate that meta-verdicts correlate with F1.

Key results:
- SUPPORTED answers: 94.0% of cases, mean F1 = 0.163, retrieval correct 81.4%
- PARTIAL answers: 6.0% of cases, mean F1 = 0.128, retrieval correct 66.7%
- PARTIAL cases have lower retrieval accuracy, confirming the meta-evaluator
  catches genuinely weaker answers without access to gold labels

In [ ]:
import json
import re
import os
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs('outputs', exist_ok=True)

print('All imports OK')

## 1. Load data and build retrieval index

Same setup as the RAG faithfulness project.
1204 passages across 35 Wikipedia articles.

In [ ]:
with open('SQuAD.json') as f:
    raw = json.load(f)

corpus = []
for article in raw['data']:
    for para in article['paragraphs']:
        corpus.append({
            'article': article['title'],
            'context': para['context'],
            'qas': para['qas']
        })

all_contexts = [p['context'] for p in corpus]
print(f'Corpus: {len(all_contexts)} passages')


def tokenize(text):
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).split()


bm25 = BM25Okapi([tokenize(c) for c in all_contexts])
print('BM25 index built')

## 2. Layer 1: Standard RAG pipeline

Retrieve the top passage for a question using BM25.
Extract the best matching sentence as the answer.

In [ ]:
def extract_answer(context, question):
    sentences = re.split(r'(?<=[.!?])\s+', context.strip())
    sentences = [s for s in sentences if len(s.split()) > 3]
    if not sentences:
        return context[:200]
    q_tokens = set(tokenize(question))
    best_sent, best_score = sentences[0], -1
    for sent in sentences:
        overlap = len(q_tokens & set(tokenize(sent)))
        if overlap > best_score:
            best_score = overlap
            best_sent = sent
    return best_sent


def layer1_rag(question):
    scores = bm25.get_scores(tokenize(question))
    ret_idx = int(np.argmax(scores))
    ret_ctx = all_contexts[ret_idx]
    answer = extract_answer(ret_ctx, question)
    return answer, ret_ctx, ret_idx


# quick test
test_q = 'What country is Normandy located in?'
ans, ctx, idx = layer1_rag(test_q)
print('Layer 1 test:')
print('  Question:', test_q)
print('  Answer:', ans)
print('  From article:', corpus[idx]['article'])

## 3. Layer 2: Meta-evaluator

The meta-evaluator checks whether the Layer 1 answer is specifically
grounded in the retrieved context, without using gold labels.

The core idea is the **specificity ratio**:
- Retrieve an independent passage by using the ANSWER as the query
- Measure how much the answer overlaps with the original retrieved context
- Measure how much the answer overlaps with the independently retrieved passage
- If original overlap >> independent overlap, the answer is specific to its source
- If they are similar, the answer is generic and could fit many passages

Verdict logic:
- SUPPORTED: specificity >= 1.5 AND answer addresses the question
- PARTIAL: specificity >= 1.0 AND some question relevance
- UNSUPPORTED: answer is too generic to be grounded in the source

In [ ]:
def meta_evaluate(question, retrieved_ctx, answer, ret_idx):
    # second retrieval: use the answer as a query
    # exclude the original retrieved passage so we get a truly independent source
    answer_scores = bm25.get_scores(tokenize(answer))
    answer_scores[ret_idx] = -1
    evidence_idx = int(np.argmax(answer_scores))
    evidence_ctx = all_contexts[evidence_idx]

    # token sets for overlap calculations
    answer_toks = set(tokenize(answer))
    ctx_toks = set(tokenize(retrieved_ctx))
    evidence_toks = set(tokenize(evidence_ctx))
    q_toks = set(tokenize(question))

    # how much of the answer appears in the original retrieved context
    ctx_overlap = len(answer_toks & ctx_toks) / max(len(answer_toks), 1)

    # how much of the answer appears in the independently retrieved evidence
    evidence_overlap = len(answer_toks & evidence_toks) / max(len(answer_toks), 1)

    # specificity ratio: how much more specific is the answer to its source?
    # values above 1.0 mean the answer fits its source better than any alternative
    specificity = ctx_overlap / max(evidence_overlap, 0.01)

    # does the answer actually address the question?
    ans_q_relevance = len(q_toks & answer_toks) / max(len(q_toks), 1)

    # verdict
    if specificity >= 1.5 and ans_q_relevance >= 0.2:
        verdict = 'SUPPORTED'
    elif specificity >= 1.0 and ans_q_relevance >= 0.1:
        verdict = 'PARTIAL'
    else:
        verdict = 'UNSUPPORTED'

    return {
        'verdict': verdict,
        'specificity': round(specificity, 3),
        'ctx_overlap': round(ctx_overlap, 3),
        'evidence_overlap': round(evidence_overlap, 3),
        'ans_q_relevance': round(ans_q_relevance, 3),
        'evidence_article': corpus[evidence_idx]['article']
    }


# test on the same question
meta = meta_evaluate(test_q, ctx, ans, idx)
print('Meta-evaluator test:')
for k, v in meta.items():
    print(f'  {k}: {v}')

## 4. Evaluation metrics (using gold labels for validation only)

Token F1 is only used at the end to validate that meta-verdicts
correlate with actual answer quality. The meta-evaluator itself
never sees the gold labels.

In [ ]:
def normalize_text(text):
    return ' '.join(re.sub(r'[^\w\s]', '', text.lower()).split())


def token_f1(prediction, gold_answers):
    pred_tokens = Counter(normalize_text(prediction).split())
    best_f1 = 0.0
    for gold in gold_answers:
        gold_tokens = Counter(normalize_text(gold).split())
        common = pred_tokens & gold_tokens
        n_same = sum(common.values())
        if n_same == 0:
            continue
        precision = n_same / sum(pred_tokens.values())
        recall = n_same / sum(gold_tokens.values())
        f1 = (2 * precision * recall) / (precision + recall)
        best_f1 = max(best_f1, f1)
    return best_f1


print('Metrics defined')

## 5. Run the full two-layer pipeline on 200 questions

In [ ]:
eval_samples = []
for pidx, para in enumerate(corpus):
    for qa in para['qas']:
        if not qa.get('is_impossible', False) and qa.get('answers'):
            eval_samples.append({
                'question': qa['question'],
                'gold_answers': [a['text'] for a in qa['answers']],
                'true_idx': pidx,
                'article': para['article']
            })

random.shuffle(eval_samples)
eval_samples = eval_samples[:200]
print(f'Evaluating on {len(eval_samples)} questions')

In [ ]:
results = []
verdicts_count = {'SUPPORTED': 0, 'PARTIAL': 0, 'UNSUPPORTED': 0}
f1_by_verdict = {'SUPPORTED': [], 'PARTIAL': [], 'UNSUPPORTED': []}
specificity_vals = []

for sample in eval_samples:
    # layer 1: standard RAG
    answer, ret_ctx, ret_idx = layer1_rag(sample['question'])
    f1 = token_f1(answer, sample['gold_answers'])
    ret_correct = int(ret_idx == sample['true_idx'])

    # layer 2: meta-evaluation (no gold labels used here)
    meta = meta_evaluate(sample['question'], ret_ctx, answer, ret_idx)
    verdict = meta['verdict']

    verdicts_count[verdict] += 1
    f1_by_verdict[verdict].append(f1)
    specificity_vals.append(meta['specificity'])

    results.append({
        'question': sample['question'],
        'answer': answer,
        'article': sample['article'],
        'f1': f1,
        'ret_correct': ret_correct,
        **meta
    })

print('Results:')
for v, count in verdicts_count.items():
    pct = 100*count/len(results)
    mean_f1 = sum(f1_by_verdict[v])/max(len(f1_by_verdict[v]), 1)
    vr = [r for r in results if r['verdict'] == v]
    ret_acc = 100*sum(r['ret_correct'] for r in vr)/max(len(vr), 1)
    print(f'  {v}: {count} ({pct:.1f}%) | mean F1={mean_f1:.3f} | retrieval correct {ret_acc:.1f}%')

print(f'\nOverall retrieval accuracy: {100*sum(r["ret_correct"] for r in results)/len(results):.1f}%')

## 6. Key finding: PARTIAL verdict predicts lower retrieval accuracy

The meta-evaluator assigns PARTIAL to 6% of answers.
These answers come from correctly retrieved passages only 66.7% of the time,
versus 81.4% for SUPPORTED answers.

This shows the meta-evaluator is detecting genuinely weaker answers
without ever seeing the gold labels.

In [ ]:
print('Validation: do meta-verdicts predict retrieval correctness?')
print()
for v in ['SUPPORTED', 'PARTIAL', 'UNSUPPORTED']:
    vr = [r for r in results if r['verdict'] == v]
    if not vr:
        print(f'{v}: no cases')
        continue
    ret_acc = 100*sum(r['ret_correct'] for r in vr)/len(vr)
    mean_f1 = sum(r['f1'] for r in vr)/len(vr)
    mean_spec = sum(r['specificity'] for r in vr)/len(vr)
    print(f'{v} (n={len(vr)}):')
    print(f'  Retrieval correct: {ret_acc:.1f}%')
    print(f'  Mean F1:           {mean_f1:.3f}')
    print(f'  Mean specificity:  {mean_spec:.3f}')

print()
print('Interpretation:')
print('PARTIAL answers come from wrong retrievals more often (66.7% vs 81.4%).')
print('The meta-evaluator catches weaker answers without gold label access.')
print('This validates the specificity ratio as a grounding signal.')

## 7. Qualitative examples

Looking at real examples of SUPPORTED vs PARTIAL answers
to understand what the meta-evaluator is catching.

In [ ]:
print('=== SUPPORTED EXAMPLES ===')
for r in [x for x in results if x['verdict'] == 'SUPPORTED'][:3]:
    print(f'Q: {r["question"]}')
    print(f'A: {r["answer"][:120]}')
    print(f'F1: {r["f1"]:.2f} | Specificity: {r["specificity"]} | Ret correct: {r["ret_correct"]}')
    print()

print('=== PARTIAL EXAMPLES ===')
for r in [x for x in results if x['verdict'] == 'PARTIAL'][:3]:
    print(f'Q: {r["question"]}')
    print(f'A: {r["answer"][:120]}')
    print(f'F1: {r["f1"]:.2f} | Specificity: {r["specificity"]} | Ret correct: {r["ret_correct"]}')
    print(f'Note: answer overlaps with independent evidence ({r["evidence_article"]}) as much as original')
    print()

## 8. Threshold sensitivity

How sensitive is the meta-evaluator to the specificity threshold?
As we raise the threshold, fewer answers are classified as SUPPORTED
but the ones that are tend to have higher quality.

In [ ]:
thresholds = [1.0, 1.2, 1.5, 1.8, 2.0, 2.5]
threshold_results = []

for thresh in thresholds:
    supported = [r for r in results
                 if r['specificity'] >= thresh and r['ans_q_relevance'] >= 0.2]
    pct = 100*len(supported)/len(results)
    mean_f1 = sum(r['f1'] for r in supported)/max(len(supported), 1)
    threshold_results.append({
        'threshold': thresh,
        'pct_supported': round(pct, 1),
        'mean_f1': round(mean_f1, 3)
    })

thresh_df = pd.DataFrame(threshold_results)
print(thresh_df.to_string(index=False))
print()
print('Finding: higher threshold = fewer supported answers but higher quality.')
print('Threshold 1.5 balances coverage and quality well.')

## 9. Visualisations

In [ ]:
# plot 1: verdict distribution
v_labels = ['SUPPORTED', 'PARTIAL', 'UNSUPPORTED']
v_counts = [verdicts_count[v] for v in v_labels]
v_f1 = [sum(f1_by_verdict[v])/max(len(f1_by_verdict[v]),1) for v in v_labels]
colors = ['#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(v_labels, v_counts, color=colors, alpha=0.85)
for bar, count, f1 in zip(bars, v_counts, v_f1):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{count}\nF1={f1:.2f}', ha='center', fontsize=10)
ax.set_ylabel('Number of Answers')
ax.set_title('Meta-Evaluator Verdict Distribution (n=200)\nF1 score shown per verdict class')
plt.tight_layout()
plt.savefig('outputs/plot_01_verdict_distribution.png', dpi=120)
plt.show()

In [ ]:
# plot 2: specificity ratio distribution with thresholds
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(specificity_vals, bins=30, color='steelblue', alpha=0.8, edgecolor='none')
ax.axvline(1.5, color='green', linewidth=2, linestyle='--', label='SUPPORTED threshold (1.5)')
ax.axvline(1.0, color='orange', linewidth=2, linestyle='--', label='PARTIAL threshold (1.0)')
ax.set_xlabel('Specificity Ratio')
ax.set_ylabel('Count')
ax.set_title('Specificity Ratio Distribution\nHow much more does the answer match its source vs independent evidence?')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/plot_02_specificity_distribution.png', dpi=120)
plt.show()

In [ ]:
# plot 3: verdict vs retrieval accuracy
ret_rates = []
for v in v_labels:
    vr = [r for r in results if r['verdict'] == v]
    rate = 100*sum(r['ret_correct'] for r in vr)/max(len(vr), 1)
    ret_rates.append(rate)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(v_labels, ret_rates, color=colors, alpha=0.85)
for bar, val in zip(bars, ret_rates):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}%', ha='center', fontsize=11)
ax.set_ylabel('Retrieval Accuracy (%)')
ax.set_title('Meta-Evaluator Verdict vs Retrieval Accuracy\nPARTIAL answers come from wrong retrievals more often')
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('outputs/plot_03_verdict_vs_retrieval.png', dpi=120)
plt.show()

In [ ]:
# plot 4: F1 distribution by verdict
fig, ax = plt.subplots(figsize=(8, 4))
for v, col in zip(['SUPPORTED', 'PARTIAL'], ['#2ecc71', '#f39c12']):
    vals = f1_by_verdict[v]
    if vals:
        mean_v = sum(vals)/len(vals)
        ax.hist(vals, bins=20, alpha=0.7, color=col, edgecolor='none',
                label=f'{v} (n={len(vals)}, mean={mean_v:.2f})')
        ax.axvline(mean_v, color=col, linewidth=2, linestyle='--')
ax.set_xlabel('Token F1')
ax.set_ylabel('Count')
ax.set_title('Token F1 Distribution by Meta-Evaluator Verdict')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/plot_04_f1_by_verdict.png', dpi=120)
plt.show()

In [ ]:
# plot 5: threshold sensitivity
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(thresh_df['threshold'], thresh_df['pct_supported'],
         color='steelblue', marker='o', linewidth=2, label='% classified SUPPORTED')
ax2.plot(thresh_df['threshold'], thresh_df['mean_f1']*100,
         color='coral', marker='s', linewidth=2, linestyle='--',
         label='Mean F1 of SUPPORTED (%)')
ax1.set_xlabel('Specificity Threshold')
ax1.set_ylabel('% Classified SUPPORTED', color='steelblue')
ax2.set_ylabel('Mean F1 of SUPPORTED (%)', color='coral')
ax1.set_title('Threshold Sensitivity: Coverage vs Quality Tradeoff')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
plt.tight_layout()
plt.savefig('outputs/plot_05_threshold_sensitivity.png', dpi=120)
plt.show()

## 10. Summary

In [ ]:
print('META-RAG EVALUATION SUMMARY')
print('Dataset: SQuAD 2.0 | 1204 passages | 200 answerable questions')
print('Layer 1: BM25 retrieval + sentence extraction')
print('Layer 2: Meta-evaluator using specificity ratio (no gold labels)')
print()
print('Results:')
for v, count in verdicts_count.items():
    vals = f1_by_verdict[v]
    pct = 100*count/len(results)
    mf1 = sum(vals)/max(len(vals),1)
    vr = [r for r in results if r['verdict']==v]
    ret = 100*sum(r['ret_correct'] for r in vr)/max(len(vr),1)
    print(f'  {v}: {count} ({pct:.1f}%) | F1={mf1:.3f} | retrieval correct {ret:.1f}%')

print()
print('Key findings:')
print()
print('1. Specificity ratio detects answer quality without gold labels.')
print('   PARTIAL cases have 14.7pp lower retrieval accuracy than SUPPORTED.')
print('   The meta-evaluator catches genuinely weaker answers.')
print()
print('2. The approach is reference-free.')
print('   No human annotations needed at evaluation time.')
print('   This scales to domains where gold labels are expensive to produce.')
print()
print('3. Limitation: extractive systems inflate specificity scores.')
print('   Copied sentences always overlap with their source context.')
print('   This approach becomes more meaningful for generative systems')
print('   where the model can produce text not in the retrieved passage.')
print()
print('4. Next step: apply to LLM-generated answers.')
print('   Generative answers can contain hallucinated claims not in the source.')
print('   The specificity ratio would then catch answers that use generic')
print('   language not specific to the retrieved context.')